In [1]:
import pyspark

In [2]:
pyspark.__file__

'/home/chris/spark/spark-3.3.2-bin-hadoop3/python/pyspark/__init__.py'

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/02 11:59:23 WARN Utils: Your hostname, desktop resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/02 11:59:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/02 11:59:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [16]:
spark.version

'3.3.2'

In [13]:
df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [12]:
!head taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


In [14]:
df.write.parquet('zones')

In [15]:
!ls -lrt

total 24
-rw-r--r-- 1 chris chris 12331 Feb 22  2024 taxi_zone_lookup.csv
-rw-r--r-- 1 chris chris  2022 Mar  2 11:58 test.ipynb
drwxr-xr-x 2 chris chris  4096 Mar  2 12:02 zones


In [95]:
df = spark.read \
    .parquet('yellow_tripdata_2024-10.parquet')

In [26]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [28]:
from pyspark.sql import types

In [40]:
schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True), 
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True), 
    types.StructField('passenger_count', types.LongType(), True), 
    types.StructField('trip_distance', types.DoubleType(), True), 
    types.StructField('RatecodeID', types.LongType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('payment_type', types.LongType(), True), 
    types.StructField('fare_amount', types.DoubleType(), True), 
    types.StructField('extra', types.DoubleType(), True), 
    types.StructField('mta_tax', types.DoubleType(), True), 
    types.StructField('tip_amount', types.DoubleType(), True), 
    types.StructField('tolls_amount', types.DoubleType(), True), 
    types.StructField('improvement_surcharge', types.DoubleType(), True), 
    types.StructField('total_amount', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True), 
    types.StructField('Airport_fee', types.DoubleType(), True)
])

In [65]:
df = spark.read \
    .schema(schema) \
    .parquet('yellow_tripdata_2024-10.parquet')

In [66]:
df.select('tpep_pickup_datetime').filter(df.tpep_pickup_datetime).show()

TypeError: DataFrame.filter() missing 1 required positional argument: 'condition'

In [96]:
from pyspark.sql import functions as F

In [75]:
df \
    .withColumn('pickup_date', F.to_date('tpep_pickup_datetime')) \
    .select('pickup_date') \
    .filter(df.pickup_date == '2024-10-15') \
    .show()
    # .filter(df.pickup_date == '2024-10-15') \
    # .count() \
    # .show()

AttributeError: 'DataFrame' object has no attribute 'pickup_date'

In [81]:
modified_df = modified_df \
    .filter(modified_df.pickup_date == "2024-10-15")
    # .withColumn("pickup_date",F.to_date('tpep_pickup_datetime')) \

In [82]:
modified_df.count()

128909

In [99]:
hour_diff = df.withColumn('hours_diff', df.tpep_dropoff_datetime - df.tpep_pickup_datetime)

In [100]:
hour_diff.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|          hours_diff|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+
|       2| 2024-09-30 20:30:44|  2024-09-30 20:48:26|              1|          3.0|         1|                 N|         16

In [102]:
hour_diff.select('hours_diff').orderBy('hours_diff', ascending=False).show(n=1)

+--------------------+
|          hours_diff|
+--------------------+
|INTERVAL '6 18:37...|
+--------------------+
only showing top 1 row



In [103]:
taxi_zones = spark.read \
    .parquet('zones')

In [105]:
taxi_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [107]:
taxi_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [109]:
df_locations = df.join(taxi_zones, df.PULocationID == taxi_zones.LocationID)

In [110]:
df_locations.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+---------+--------------------+------------+
|       2| 2024-09-30 

In [112]:
df_locations.groupBy("Zone").agg({"tpep_pickup_datetime": "count"}).sort("count(tpep_pickup_datetime)").show()

+--------------------+---------------------------+
|                Zone|count(tpep_pickup_datetime)|
+--------------------+---------------------------+
|Governor's Island...|                          1|
|       Arden Heights|                          2|
|       Rikers Island|                          2|
|         Jamaica Bay|                          3|
| Green-Wood Cemetery|                          3|
|Charleston/Totten...|                          4|
|       Port Richmond|                          4|
|   Rossville/Woodrow|                          4|
|Eltingville/Annad...|                          4|
|       West Brighton|                          4|
|        Crotona Park|                          6|
|         Great Kills|                          6|
|Heartland Village...|                          7|
|     Mariners Harbor|                          7|
|Saint George/New ...|                          9|
|             Oakwood|                          9|
|New Dorp/Midland ...|         